# k-medoids on the STL-10 embeddings

Same data, same `K = 10`

* the centre of a cluster must be **one of the observed images** thus a *medoid*,
* the objective sums **distances** to the nearest medoid, not squared distances to a mean.

scikit-learn has no k-medoids, so `pam.py` is needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import normalized_mutual_info_score, silhouette_score

DATA = "../week_2/data"
IMG = 96

d = np.load(f"{DATA}/embeddings_stl10_1000.npz")

X = d["E_l12"]
y = d["y"]

class_names = [l.strip() for l in open(f"{DATA}/stl10_binary/class_names.txt") if l.strip()]

def load_rgb(split):
    A = np.fromfile(f"{DATA}/stl10_binary/{split}_X.bin", dtype=np.uint8).reshape(-1, 3, IMG, IMG)
    return A.transpose(0, 3, 2, 1)

X_img = np.concatenate([load_rgb("train"), load_rgb("test")])[d["idx"]]

K = 10
RANDOM_STATE = 0

print(X.shape, X_img.shape, y.shape)

### k-medoids with K = 10

In [ ]:
import time

from pam import pam, labels as pam_labels

t0 = time.perf_counter()
medoids, cost = pam(X, K, max_iter=300, random_state=RANDOM_STATE)
labels = pam_labels(X, medoids)
seconds = time.perf_counter() - t0

print("cluster sizes:", np.bincount(labels, minlength=K))
print(f"cost {cost:.1f}   {seconds:.1f} s")

In [ ]:
def report(name, labels):
    print(f"{name}")
    print(f"  NMI        {normalized_mutual_info_score(y, labels):.3f}")
    print(f"  silhouette {silhouette_score(X, labels):.3f}")

### The medoids are real images

This is the property k-means does not have: every cluster centre is an actual photograph
from the dataset, so we can simply display it.

In [ ]:
fig, axes = plt.subplots(1, K, figsize=(15, 1.9))

for ax, m in zip(axes, medoids):
    ax.imshow(X_img[m])
    ax.set_title(f"{class_names[y[m] - 1]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

### What ended up in each cluster?

In [ ]:
pd.crosstab(pd.Series([class_names[v - 1] for v in y], name="class"),
            pd.Series(labels, name="cluster"))